# GrowthParameterEstimation Function Tour

This notebook is a guided walkthrough of the package APIs.
It includes: data layer, exposure profiles, model registry/simulation, fitting, analysis, and pipeline workflow.

In [26]:
using Pkg
project_root = normpath(joinpath(@__DIR__, ".."))
Pkg.activate(project_root)

try
    @eval using GrowthParameterEstimation
catch
    println("Package load failed on first try; running Pkg.instantiate()...")
    Pkg.instantiate()
    @eval using GrowthParameterEstimation
end

using DataFrames, CSV, Random
using OrdinaryDiffEq, DifferentialEquations

Random.seed!(42)
x = collect(0.0:1.0:8.0)
y = [1.0, 1.6, 2.4, 3.1, 3.9, 4.6, 5.2, 5.7, 6.0]
p0 = [0.2, 12.0]
bounds = [(0.01, 2.0), (2.0, 100.0)]
println("Setup complete with project: ", project_root)

  Activating project at `d:\3_Research\GrowthParameterEstimation.jl`


Setup complete with project: d:\3_Research\GrowthParameterEstimation.jl\


In [27]:
# Data + Exposure
raw = DataFrame(time=x, count=y, dose=fill(0.2, length(x)), cell_line=fill("A549", length(x)), density=fill(1.0, length(x)), replicate=fill(1, length(x)))
norm = normalize_schema(raw)
@assert validate_timeseries(norm)

exp_constant = build_exposure(:constant; value=0.25)
exp_pulse = build_exposure(:pulse; amplitude=1.0, start_time=2.0, end_time=4.0)
exp_vals = evaluate_exposure(exp_pulse, x)
println("Rows: ", nrow(norm), " | Pulse values: ", exp_vals)

Rows: 9 | Pulse values: [0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0]


In [28]:
# Registry + Simulation + Observation
models = list_models()
spec = get_model("logistic_growth")
sim = simulate(spec, x, [0.4, 20.0]; u0=[y[1]], exposure=ConstantExposure(0.2))
obs_spec = ObservationSpec("viable_scaled", viable_total, 1.2, 0.1)
obs_val = observed_signal(obs_spec, [3.0], nothing, 0.0)
println("Models: ", length(models), " | Sim success: ", sim.success, " | Observed signal: ", obs_val)

Models: 11 | Sim success: true | Observed signal: 3.5999999999999996


In [29]:
# Fitting + comparisons
fit = run_single_fit(x, y, p0; model=logistic_growth!, solver=Tsit5(), bounds=bounds, show_stats=false)
comp = compare_models(x, y, "Logistic", logistic_growth!, [0.2, 12.0], "Gompertz", gompertz_growth!, [0.2, 1.2, 12.0];
    solver=Tsit5(), bounds1=bounds, bounds2=[(0.01, 2.0), (0.1, 5.0), (2.0, 100.0)], show_stats=false, output_csv=joinpath(tempdir(), "nb_compare.csv"))

specs = Dict("Logistic" => (model=logistic_growth!, p0=[0.2, 12.0], bounds=bounds),
             "Exp" => (model=exponential_growth!, p0=[0.2], bounds=[(0.001, 2.0)]))
_ = compare_models_dict(x, y, specs; default_solver=Tsit5(), show_stats=false, output_csv=joinpath(tempdir(), "nb_compare_dict.csv"))
println("Fit params: ", fit.params, " | Best model: ", comp.best_model.name)

Row,Model,BIC
,String,Float64
1,Logistic,-44.5846
2,Exp,-0.565444


=== Logistic ===
Params: [0.536975031161508, 6.387403427636794], BIC: -44.58455542716798, SSR: 0.038972966383158114
=== Gompertz ===
Params: [0.28263422962377693, 2.550000000006081, 7.470768118995889], BIC: -52.98361690245157, SSR: 0.012007159767496015
Results saved to C:\Users\elbak\AppData\Local\Temp\nb_compare.csv

BIC Summary:
Summary saved to C:\Users\elbak\AppData\Local\Temp\nb_compare_dict.csv
Predictions saved to C:\Users\elbak\AppData\Local\Temp\nb_compare_dict.csv
Fit params: [0.536975031161508, 6.387403427636794] | Best model: Gompertz


In [30]:
# Analysis
loo = leave_one_out_validation(x, y, [0.2, 12.0]; model=logistic_growth!, solver=Tsit5(), bounds=bounds, show_stats=false)
kfold = k_fold_cross_validation(x, y, [0.2, 12.0]; k_folds=3, model=logistic_growth!, solver=Tsit5(), bounds=bounds, show_stats=false)
sens = parameter_sensitivity_analysis(x, y, fit; perturbation=0.1, model=logistic_growth!, solver=Tsit5())
resid = residual_analysis(x, y, fit; model=logistic_growth!, solver=Tsit5())
enh = enhanced_bic_analysis(x, y; models=[logistic_growth!, gompertz_growth!], model_names=["Logistic", "Gompertz"], p0_values=[[0.2, 12.0], [0.2, 1.2, 12.0]], solver=Tsit5())
println("LOO RMSE: ", loo.rmse, " | k-fold RMSE: ", kfold.overall_rmse, " | Best enhanced BIC model: ", enh.best_model.model_name)

Performing leave-one-out cross-validation...
Performing 3-fold cross-validation...
Performing parameter sensitivity analysis...
Base parameters: [0.537, 6.3874]
Perturbation: ±10.0%
Parameter 1: SI = 0.854, Max rel change = 8.54%
Parameter 2: SI = 0.926, Max rel change = 9.26%

=== Parameter Sensitivity Ranking ===
1. Parameter 2 (value=6.3874): SI=0.926
2. Parameter 1 (value=0.537): SI=0.854
=== Residual Analysis Summary ===
RMSE: 0.0658
MAE: 0.051
Mean residual: 0.0178
Std residual: 0.0672
Max |residual|: 0.1514
Outliers (|std_resid| > 2.0): 1
Normality correlation: 0.974
Normality concern: No
Durbin-Watson statistic: 1.038
Autocorrelation concern: Yes

Outlier details:
  Point 3: x=2.0, y=2.4, residual=0.151, std_resid=2.253
=== Enhanced BIC Analysis ===
Comparing 2 models...

Fitting Logistic...
  Success: BIC=-44.58, R²=0.9985, RMSE=0.0658

Fitting Gompertz...
  Success: BIC=-52.98, R²=0.9995, RMSE=0.0365

=== Model Comparison Results ===
Model Rankings by BIC (lower is better):
1

In [31]:
# Workflow pipeline (robust plotting even if some model fits fail)
try
    @eval using Plots
catch
    println("Plots not found; installing into active project...")
    Pkg.add("Plots")
    @eval using Plots
end

df_work = DataFrame(
    time = vcat(x, x),
    count = vcat(y, y .* 0.9),
    error = fill(0.2, 2 * length(x)),
    dose = vcat(fill(0.2, length(x)), fill(0.6, length(x))),
    cell_line = fill("A549", 2 * length(x)),
    density = fill(1.0, 2 * length(x)),
    replicate = vcat(fill(1, length(x)), fill(2, length(x))),
    unit_time = fill("h", 2 * length(x)),
    unit_count = fill("count", 2 * length(x)),
)

top_k_models = 2  # number of model curves overlaid per condition plot
fig_dir = joinpath(tempdir(), "nb_results", "figures")
mkpath(fig_dir)

conds = build_conditions(df_work)
# Use robust baseline models and slightly stronger optimizer settings for notebook reliability
ranked = rank_models(["logistic_growth", "gompertz_growth"], conds; n_starts=4, maxiters=120, top_k=top_k_models, seed=42)

# Keep only successfully fitted models
success_models = [m for m in ranked.ranking.model if haskey(ranked.fits, m)]
selected_models = success_models[1:min(top_k_models, length(success_models))]

if isempty(selected_models)
    println("No successful model fits available for plotting (see ranked.failures).")
else
    default(size=(1200, 700), lw=2.5, guidefontsize=12, tickfontsize=10, legendfontsize=10)

    # Overlay prediction plots by condition
    for cond in conds
        p = scatter(cond.time, cond.count; label="Observed", xlabel="Time", ylabel="Count", title=cond.name, legend=:topleft, markersize=4)
        for model_name in selected_models
            fit_info = ranked.fits[model_name]
            idx = findfirst(pc -> pc.condition == cond.name, fit_info.per_condition)
            if !isnothing(idx)
                plot!(p, cond.time, fit_info.per_condition[idx].observed; label=model_name)
            end
        end
        safe_name = replace(cond.name, '|' => '_', ' ' => '_')
        savefig(p, joinpath(fig_dir, safe_name * "_overlay.png"))
        display(p)
    end

    # BIC bar chart
    bic_vals = [ranked.ranking[findfirst(==(m), ranked.ranking.model), :bic] for m in selected_models]
    p_bic = bar(selected_models, bic_vals; legend=false, xlabel="Model", ylabel="BIC", title="Top Model BIC Comparison", linecolor=:black, linewidth=0.8, bar_width=0.6)
    savefig(p_bic, joinpath(fig_dir, "top_models_bic.png"))
    display(p_bic)
end

# Optional exports (guarded against failed-model edge cases)
try
    _ = export_results(ranked; output_dir=joinpath(tempdir(), "nb_results"))
catch err
    println("export_results skipped: ", err)
end

cfg = default_config(output_dir=joinpath(tempdir(), "nb_pipeline"))
cfg = PipelineConfig(cfg.version, cfg.model_names, 4, top_k_models, 120, cfg.reltol, cfg.abstol, cfg.weighted, cfg.seed, cfg.output_dir)
cfg_path = save_config(joinpath(tempdir(), "nb_config.toml"), cfg)
cfg_loaded = load_config(cfg_path)
pipe = run_pipeline(df_work; config=cfg_loaded, include_models=["logistic_growth", "gompertz_growth"])

println("Ranking rows: ", nrow(pipe.ranking), " | Selected models plotted: ", length(selected_models))
println("Figures saved under: ", fig_dir)
println("BIC plot: ", joinpath(fig_dir, "top_models_bic.png"))

No successful model fits available for plotting.
export_results skipped: KeyError("logistic_growth")


KeyError: KeyError: key "logistic_growth" not found

In [32]:
# Full console-style run (same as script)
include(joinpath(@__DIR__, "api_tour.jl"))

Row,Model,BIC
,String,Float64
1,Logistic,-44.5846
2,Exp,-0.565444


GrowthParameterEstimation API Tour

[1/8] Data layer
- REQUIRED_COLUMNS: [:time, :count, :error, :dose, :cell_line, :density, :replicate, :unit_time, :unit_count]
- Loaded rows: 9

[2/8] Exposure layer
- Constant exposure at t=2: 0.25
- Pulse profile: [0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0]
- Step profile: [0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.1, 0.1, 0.1]
- Decay profile: [0.0, 1.0, 0.67, 0.449, 0.301, 0.202, 0.135, 0.091, 0.061]

[3/8] Registry + simulation
- Registered models: 11
- First few models: ["adaptive_ic50", "bi_exponential_response", "damage_repair_arrest", "gompertz_growth", "logistic_growth"]
- Sim success: true | observed points: 9
- Custom model registered: true

[4/8] Observation helpers
- observed_signal: 3.5999999999999996
- sum_states([1,2]): 5.0

[5/8] Fitting helpers
- Baseline BIC/SSR: 14.808 / 28.624
- run_single_fit params: [0.537, 6.3874]
Optimized params: [0.536975031161508, 6.387403427636794]
SSR: 0.038972966383158114
BIC: -44.58455542716798

[6/8] Compari

LoadError: LoadError: KeyError: key "logistic_growth" not found
in expression starting at d:\3_Research\GrowthParameterEstimation.jl\tests\api_tour.jl:152